In [1]:
from src.module.database import oracle_import, oracle_export, oracle_execute
import json
import os
import sys
from datetime import datetime, timedelta
from sqlalchemy import create_engine, text

In [2]:
oracle_execute("grant select on uni_ml_report.credit_collection_score to uni_ml_scoring")

2026-08-07 15:42:14.762 | INFO     | __main__:<module>:1 - Function oracle_execute executed in: 145 ms


In [3]:
p_date = str(datetime.today().date() -timedelta(days = 3)).replace("-", "")
p_date

'20260804'

In [4]:
consumer_activities = oracle_import(f"select * from toki.marketplace_consumer_activities where p_date = '{p_date}'")

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-08-07 15:42:25.390 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 8 sec 684 ms


In [5]:
consumer_activities["ACTIVITYNAME"].unique()

<StringArray>
['cart-events', 'limit-events', 'order-events', 'wishlist-events']
Length: 4, dtype: str

In [6]:
consumer_activities.groupby("ACTIVITYNAME").count()

,ID_,ACTIVITYDATA,CREATEDAT,UPDATEDAT,P_DATE
ACTIVITYNAME,,,,,
cart-events,144761,144761,144761,144761,144761
limit-events,834,834,834,834,834
order-events,502,502,502,502,502
wishlist-events,56,56,56,56,56


In [7]:
consumer_activities.tail()

,ID_,ACTIVITYNAME,ACTIVITYDATA,CREATEDAT,UPDATEDAT,P_DATE
146148,6a71f35e4aeec35317296bb2,limit-events,"{'accountId': '62dcd75114558aa7f6c3c526', 'res...",2026-08-04 22:12:46,2026-08-04 22:12:46,20260804
146149,6a71f36d805151b025f64387,order-events,"{'type': 'ORDER_EXPIRED', 'data': {'accountId'...",2026-08-04 22:13:01,2026-08-04 22:13:01,20260804
146150,6a71f39e420fe633e049db4a,cart-events,"{'cartId': '68330c0549783601357d69f2', 'type':...",2026-08-04 22:13:50,2026-08-04 22:13:50,20260804
146151,6a71f39f805151b025f6438f,cart-events,"{'cartId': '678fe3d9733c6bf8bebb1fa5', 'type':...",2026-08-04 22:13:51,2026-08-04 22:13:51,20260804
146152,6a71f3a7420fe633e049db4e,limit-events,"{'accountId': '65054836e4f0bda78f2f2767', 'res...",2026-08-04 22:13:59,2026-08-04 22:13:59,20260804


In [8]:
consumer_activities["ACTIVITYDATA"]  = consumer_activities["ACTIVITYDATA"].apply(lambda x: str(x).replace("'", "\"").replace("None", "null").replace("True", "true").replace("False", "false") if isinstance(x, str) else x)

In [9]:
json.loads(str(consumer_activities["ACTIVITYDATA"].values[0]))

{'cartId': '636f90247601be5ba66b5e7b',
 'type': 'PRODUCT_MODIFIED',
 'item': {'productId': '68d3cd51d36b9be827b44e3c',
  'qty': 1,
  'available': True,
  '_id': '6a479f089bd8f5a236e433a9'},
 'cart': {'_id': '68dd5104175e6115c847767e',
  'accountId': '636f90247601be5ba66b5e7b',
  'items': [{'productId': '68b7ee190bcb0200c3e8d7c7',
    'qty': 1,
    'available': True,
    '_id': '6a1e65aa496155f535bfbb9b'},
   {'productId': '68d3cd51d36b9be827b44e3c',
    'qty': 1,
    'available': True,
    '_id': '6a479f089bd8f5a236e433a9'}],
  'createdAt': '2025-10-01T16:04:20.813Z',
  'updatedAt': '2026-08-04T00:03:11.930Z'}}

In [10]:
def safe_json_loads(x):
    if not isinstance(x, str):
        return x
    try:
        return json.loads(x)
    except (json.JSONDecodeError, ValueError):
        return None

consumer_activities["ACTIVITYDATA"] = consumer_activities["ACTIVITYDATA"].apply(safe_json_loads)

In [11]:
consumer_activities["ACTIVITYDATA"].values[0] #

{'cartId': '636f90247601be5ba66b5e7b',
 'type': 'PRODUCT_MODIFIED',
 'item': {'productId': '68d3cd51d36b9be827b44e3c',
  'qty': 1,
  'available': True,
  '_id': '6a479f089bd8f5a236e433a9'},
 'cart': {'_id': '68dd5104175e6115c847767e',
  'accountId': '636f90247601be5ba66b5e7b',
  'items': [{'productId': '68b7ee190bcb0200c3e8d7c7',
    'qty': 1,
    'available': True,
    '_id': '6a1e65aa496155f535bfbb9b'},
   {'productId': '68d3cd51d36b9be827b44e3c',
    'qty': 1,
    'available': True,
    '_id': '6a479f089bd8f5a236e433a9'}],
  'createdAt': '2025-10-01T16:04:20.813Z',
  'updatedAt': '2026-08-04T00:03:11.930Z'}}

In [12]:
consumer_events = oracle_import(f"select * from toki.marketplace_consumer_EVENTS where p_date ={p_date} ")

/workspaces/marketplace-stream-data-recommendation-engine/src/module/database.py:127: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data_frame = pd.read_sql(query, engine)
2026-08-07 15:44:45.570 | INFO     | __main__:<module>:1 - Function oracle_import executed in: 2 min 18 sec 


In [13]:
p_date

'20260804'

In [14]:
consumer_events.shape

(10894, 11)

In [15]:
consumer_events.groupby("EVENTNAME").count()

,ID_,EVENTVALUE,ACCOUNTID,SESSIONID,TIMESTAMP_,USERAGENT,URL_,CREATEDAT,UPDATEDAT,P_DATE
EVENTNAME,,,,,,,,,,
product_click,4782,4782,4782,4692,4782,4782,4782,4782,4782,4782
taxon_click,6112,6112,6112,5945,6112,6112,6112,6112,6112,6112


In [16]:
import config as cfg
import pandas as pd
import numpy as np
from src.database import pgsql_import

In [17]:
full_catalog_data = pgsql_import("select * from marketplace_catalog_data_extended_version3_staging")

In [18]:
full_catalog_data.shape

(4126, 52)

In [19]:
full_catalog_data.head()

,carried_located_in,main_category,sub_category,product_category,exact_product_category,manufacturer,generic_name,actual_product,size,power_consumption,...,color,image,productstate,createdat,updatedat,productmeta,saleprice,image_urls,details_translation,group_id
0,Living Room,Electronics,Televisions,Mini LED QLED 4K TVs,Sony 85XR50 85-inch Mini LED QLED 4K HDR Googl...,SONY,Smart Television,Mini LED QLED 4K HDR Google 85 inch tv /SONY-K...,85 inches,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9769a9ad-9471-5347-abf4-c920d69a0e3e
1,Living Room,Electronics,Televisions,4K UHD TVs,Full Array LED 4K HDR Smart TV,Panasonic,Smart Television,Panasonic TH-75NX900,75 inch,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,eae2fd2c-1c51-56f4-a7ec-7561dbfd11e9
2,Living Room,Electronics,Televisions,4K UHD TVs,Full Array LED TVs,Panasonic,Smart Television,Panasonic TH-55NX900 Full Array Led 55 inch sm...,55 inches,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,278b1e1a-844e-5fe0-ba34-76ea6f448aef
3,Living Room,Electronics,Televisions,Mini LED / QLED 4K TVs,Mini LED QLED 4K HDR Google TV (Sony XR90 series),SONY,Smart Television,Mini LED QLED 4K HDR Google 85 inch TV /SONY-K...,85 inches,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1840c5f5-1358-5611-8978-af4464e30d73
4,Studio,Electronics,Computer Audio,Microphones,Condenser Microphone,Sennheiser,Microphone,Sennheiser MK4 Condenser Microphone,,,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,291e484e-b9e9-5b57-8c03-7e6b549927a9


## Snapshot Analysis Direction

The goal of this notebook section is to turn yesterday's activities/events plus the full catalog into one canonical interaction dataset. That interaction table becomes the contract for a realtime consumer pipeline later: every incoming event should map to `account_id`, `product_id` or `taxonomy/category`, `event_type`, `event_ts`, `weight`, and optional context.


In [20]:
# Quick schema and volume checks
print("consumer_activities", consumer_activities.shape)
print("consumer_events", consumer_events.shape)
print("full_catalog_data", full_catalog_data.shape)

print("\nActivity columns")
display(pd.DataFrame({"column": consumer_activities.columns, "dtype": consumer_activities.dtypes.astype(str).values}))

print("\nEvent columns")
display(pd.DataFrame({"column": consumer_events.columns, "dtype": consumer_events.dtypes.astype(str).values}))

print("\nCatalog columns")
display(pd.DataFrame({"column": full_catalog_data.columns, "dtype": full_catalog_data.dtypes.astype(str).values}))


consumer_activities (146153, 6)
consumer_events (10894, 11)
full_catalog_data (4126, 52)

Activity columns


,column,dtype
0,ID_,str
1,ACTIVITYNAME,str
2,ACTIVITYDATA,object
3,CREATEDAT,datetime64[us]
4,UPDATEDAT,datetime64[us]
5,P_DATE,str



Event columns


,column,dtype
0,ID_,str
1,EVENTNAME,str
2,EVENTVALUE,str
3,ACCOUNTID,str
4,SESSIONID,str
5,TIMESTAMP_,str
6,USERAGENT,str
7,URL_,str
8,CREATEDAT,datetime64[us]
9,UPDATEDAT,datetime64[us]



Catalog columns


,column,dtype
0,carried_located_in,str
1,main_category,str
2,sub_category,str
3,product_category,str
4,exact_product_category,str
5,manufacturer,str
6,generic_name,str
7,actual_product,str
8,size,str
9,power_consumption,str


In [21]:
# Inspect nested ACTIVITYDATA keys per activity type. This tells us which fields can feed the stream contract.
def flatten_keys(value, prefix=""):
    keys = set()
    if isinstance(value, dict):
        for key, nested in value.items():
            path = f"{prefix}.{key}" if prefix else str(key)
            keys.add(path)
            keys.update(flatten_keys(nested, path))
    elif isinstance(value, list):
        for item in value[:3]:
            keys.update(flatten_keys(item, f"{prefix}[]" if prefix else "[]"))
    return keys

activity_key_profile = (
    consumer_activities
    .assign(_keys=consumer_activities["ACTIVITYDATA"].apply(flatten_keys))
    .groupby("ACTIVITYNAME")["_keys"]
    .agg(lambda rows: sorted(set().union(*rows)))
)

for activity_name, keys in activity_key_profile.items():
    print(f"\n{activity_name}: {len(keys)} keys")
    print(keys[:80])



cart-events: 23 keys
['cart', 'cart._id', 'cart.accountId', 'cart.createdAt', 'cart.items', 'cart.items[]._id', 'cart.items[].available', 'cart.items[].productId', 'cart.items[].qty', 'cart.updatedAt', 'cartId', 'item', 'item._id', 'item.accountId', 'item.available', 'item.items', 'item.items[].available', 'item.items[].productId', 'item.items[].qty', 'item.items[].storeId', 'item.productId', 'item.qty', 'type']

limit-events: 5 keys
['accountId', 'result', 'result.expireDate', 'result.limit', 'result.status']

order-events: 13 keys
['data', 'data.accountId', 'data.deliveryMethod', 'data.items', 'data.items[].name', 'data.items[].productId', 'data.items[].qty', 'data.items[].transNumber', 'data.items[].unitPrice', 'data.orderId', 'data.orderNo', 'data.vendorOrderId', 'type']

wishlist-events: 3 keys
['accountId', 'item', 'type']


In [22]:
# Normalize activity rows into account/product interactions.
def nested_get(value, path, default=None):
    current = value
    for part in path.split("."):
        if not isinstance(current, dict) or part not in current:
            return default
        current = current[part]
    return current

def normalize_activity_row(row):
    data = row.get("ACTIVITYDATA") if isinstance(row.get("ACTIVITYDATA"), dict) else {}
    activity_name = row.get("ACTIVITYNAME")
    event_type = data.get("type") or activity_name
    account_id = data.get("accountId") or nested_get(data, "cart.accountId")
    product_id = nested_get(data, "item.productId")

    if product_id is None and isinstance(nested_get(data, "cart.items"), list) and data["cart"]["items"]:
        product_id = data["cart"]["items"][0].get("productId")

    return {
        "source": "consumer_activities",
        "source_id": row.get("ID_"),
        "account_id": account_id,
        "session_id": None,
        "event_name": activity_name,
        "event_type": event_type,
        "product_id": product_id,
        "event_value": None,
        "event_ts": row.get("CREATEDAT"),
        "p_date": row.get("P_DATE"),
    }

activity_interactions = pd.DataFrame(
    normalize_activity_row(row) for _, row in consumer_activities.iterrows()
)

activity_interactions.head()


,source,source_id,account_id,session_id,event_name,event_type,product_id,event_value,event_ts,p_date
0,consumer_activities,6a712c3f805151b025f5a397,636f90247601be5ba66b5e7b,None,cart-events,PRODUCT_MODIFIED,68d3cd51d36b9be827b44e3c,None,2026-08-04 08:03:11,20260804
1,consumer_activities,6a712c3f420fe633e0493b62,5fa00c3bf98f0ee0f5c5f6db,None,cart-events,PRODUCT_MODIFIED,6768c4c357862688381fb60b,None,2026-08-04 08:03:11,20260804
2,consumer_activities,6a712c3f805151b025f5a399,5fb1581e2280fe45d16b484a,None,cart-events,PRODUCT_MODIFIED,69c644e85a6a340632dfd5c6,None,2026-08-04 08:03:11,20260804
3,consumer_activities,6a712c3f805151b025f5a39b,63aed5fc55c2deccaabfa857,None,cart-events,PRODUCT_MODIFIED,68d3cd51d36b9be827b44e3c,None,2026-08-04 08:03:11,20260804
4,consumer_activities,6a712c3f4aeec3531728ce58,6753e050061c99ac254c80c2,None,cart-events,PRODUCT_MODIFIED,6791e465eaa75841a5a8b5e1,None,2026-08-04 08:03:11,20260804


In [23]:
# Normalize click events. EVENTVALUE may be product id, taxon/category id, slug, or JSON depending on source.
def safe_event_value(value):
    if isinstance(value, dict):
        return value
    if not isinstance(value, str):
        return value
    try:
        return json.loads(value)
    except (json.JSONDecodeError, ValueError):
        return value

def normalize_event_row(row):
    value = safe_event_value(row.get("EVENTVALUE"))
    product_id = None
    taxon_id = None

    if isinstance(value, dict):
        product_id = value.get("productId") or value.get("product_id") or value.get("id")
        taxon_id = value.get("taxonId") or value.get("taxon_id") or value.get("categoryId")
    elif row.get("EVENTNAME") == "product_click":
        product_id = value
    elif row.get("EVENTNAME") == "taxon_click":
        taxon_id = value

    return {
        "source": "consumer_events",
        "source_id": row.get("ID_"),
        "account_id": row.get("ACCOUNTID"),
        "session_id": row.get("SESSIONID"),
        "event_name": row.get("EVENTNAME"),
        "event_type": row.get("EVENTNAME"),
        "product_id": product_id,
        "taxon_id": taxon_id,
        "event_value": value,
        "event_ts": row.get("TIMESTAMP_") or row.get("CREATEDAT"),
        "p_date": row.get("P_DATE"),
    }

event_interactions = pd.DataFrame(
    normalize_event_row(row) for _, row in consumer_events.iterrows()
)

event_interactions.head()


,source,source_id,account_id,session_id,event_name,event_type,product_id,taxon_id,event_value,event_ts,p_date
0,consumer_events,6a6d92748c8c31c847767923,63aaf8205cc119cc799d7e3e,edjaK5n-bHwHiah1iFJF8dFyP1DpKSm_,product_click,product_click,"{'productIds': ['68febd4c9494859a95029a58'], '...",NaN,"{'productIds': ['68febd4c9494859a95029a58'], '...",2026-08-01T06:30:11.079Z,20260801
1,consumer_events,6a6d927b805151b025f3468b,65adc2aef516d7968dac7d55,BrcdpNDjJORfdRryEP_bRlYRbc6PgVrT,product_click,product_click,"{'productIds': ['68d3cd51d36b9be827b44e3c'], '...",NaN,"{'productIds': ['68d3cd51d36b9be827b44e3c'], '...",2026-08-01T06:30:19.431Z,20260801
2,consumer_events,6a6d927dce31add3c36093be,68ba42f8a17feb29bd0d35e0,xU1c0Nco9aAsDHi55WRDK8ca3bOwP5VC,product_click,product_click,"{'productIds': ['68febd4c9494859a95029a58'], '...",NaN,"{'productIds': ['68febd4c9494859a95029a58'], '...",2026-08-01T06:30:21.380Z,20260801
3,consumer_events,6a6d927f4aeec35317267444,66495540b3f428cd2bf80d92,HBCF0jVTQnCgn5hBGJw1RELZs9mVHp_7,taxon_click,taxon_click,NaN,{'taxon': {'label': 'Тренд технологи'}},{'taxon': {'label': 'Тренд технологи'}},2026-08-01T06:30:23.221Z,20260801
4,consumer_events,6a6d9281ce31add3c36093c0,61e2adeb1c2a7cb33e51cb3a,LB1U3N-sxSRGdqYu_p49PV__g35hRzpb,taxon_click,taxon_click,NaN,{'taxon': {'label': 'Гоо сайхны хэрэгсэл'}},{'taxon': {'label': 'Гоо сайхны хэрэгсэл'}},2026-08-01T06:30:25.203Z,20260801


In [24]:
# Canonical behavior table. Tune weights after validating business outcomes.
EVENT_WEIGHTS = {
    "product_click": 1.0,
    "taxon_click": 0.3,
    "cart-events": 3.0,
    "PRODUCT_ADDED": 4.0,
    "PRODUCT_MODIFIED": 2.0,
    "wishlist-events": 5.0,
    "order-events": 10.0,
    "limit-events": 1.5,
}

interactions = pd.concat([activity_interactions, event_interactions], ignore_index=True, sort=False)
interactions["event_ts"] = pd.to_datetime(interactions["event_ts"], errors="coerce", utc=True)
interactions["weight"] = interactions["event_type"].map(EVENT_WEIGHTS).fillna(
    interactions["event_name"].map(EVENT_WEIGHTS)
).fillna(1.0)
interactions["has_product_id"] = interactions["product_id"].notna() & (interactions["product_id"].astype(str).str.len() > 0)

summary = pd.DataFrame({
    "metric": [
        "rows",
        "unique_accounts",
        "product_level_rows",
        "rows_missing_account",
        "rows_missing_event_ts",
    ],
    "value": [
        len(interactions),
        interactions["account_id"].nunique(dropna=True),
        int(interactions["has_product_id"].sum()),
        int(interactions["account_id"].isna().sum()),
        int(interactions["event_ts"].isna().sum()),
    ]
})

display(summary)
display(interactions.groupby(["source", "event_name", "event_type"], dropna=False).agg(
    rows=("source_id", "count"),
    accounts=("account_id", "nunique"),
    product_rows=("has_product_id", "sum"),
    avg_weight=("weight", "mean"),
).sort_values("rows", ascending=False).head(30))


,metric,value
0,rows,136984
1,unique_accounts,31004
2,product_level_rows,128723
3,rows_missing_account,460
4,rows_missing_event_ts,0


rows  accounts  \
source              event_name      event_type                           
consumer_activities cart-events     PRODUCT_MODIFIED  122309     28544   
consumer_events     taxon_click     taxon_click         6252      1846   
                    product_click   product_click       5302      1706   
consumer_activities limit-events    limit-events        1310       859   
                    cart-events     ITEM_ADDED           790       517   
                                    ITEM_REMOVED         271       157   
                                    PRODUCT_ORDERED      210       167   
                    order-events    PRODUCT_ORDERED      203         0   
                                    ORDER_EXPIRED        174         0   
                    wishlist-events ITEM_ADDED            64        40   
                    order-events    ORDER_ACTIVATED       34         0   
                                    ORDER_COMPLETED       24         0   
                    cart-events     cart-events           18         0   
                                    PRODUCT_REMOVED        9         9   
                    wishlist-events ITEM_REMOVED           8         7   
                    order-events    order-events           6         0   

                                                      product_rows  avg_weight  
source              event_name      event_type                                  
consumer_activities cart-events     PRODUCT_MODIFIED        122308         2.0  
consumer_events     taxon_click     taxon_click                  0         0.3  
                    product_click   product_click             5302         1.0  
consumer_activities limit-events    limit-events                 0         1.5  
                    cart-events     ITEM_ADDED                 790         3.0  
                                    ITEM_REMOVED               271         3.0  
                                    PRODUCT_ORDERED             43         3.0  
                    order-events    PRODUCT_ORDERED              0        10.0  
                                    ORDER_EXPIRED                0        10.0  
                    wishlist-events ITEM_ADDED                   0         5.0  
                    order-events    ORDER_ACTIVATED              0        10.0  
                                    ORDER_COMPLETED              0        10.0  
                    cart-events     cart-events                  0         3.0  
                                    PRODUCT_REMOVED              9         3.0  
                    wishlist-events ITEM_REMOVED                 0         5.0  
                    order-events    order-events                 0        10.0

In [25]:
# Catalog readiness check. Select the strongest available product id column before joining.
candidate_product_columns = [
    col for col in full_catalog_data.columns
    if str(col).lower() in {"product_id", "productid", "id", "_id", "group_id"}
    or "product" in str(col).lower()
]

print(candidate_product_columns)

catalog_profile = full_catalog_data.agg(["count", "nunique"]).T.reset_index()
catalog_profile.columns = ["column", "non_null_rows", "unique_values"]
display(catalog_profile.sort_values(["non_null_rows", "unique_values"], ascending=False).head(30))


['product_category', 'exact_product_category', 'actual_product', 'product_id', 'productstate', 'productmeta', 'group_id']


,column,non_null_rows,unique_values
17,index,4126,4126
18,product_id,4126,4126
24,keywords,4126,4122
22,details,4126,4121
11,specifications,4126,4117
21,main_option,4126,4045
12,sku,4126,3669
7,actual_product,4126,3664
23,url_link,4126,3620
4,exact_product_category,4126,3094


In [26]:
# User preference profile from the normalized interactions.
# After confirming the catalog join key, enrich this with category/manufacturer/price attributes.
user_behavior_profile = (
    interactions[interactions["account_id"].notna()]
    .sort_values("event_ts")
    .groupby("account_id")
    .agg(
        total_events=("source_id", "count"),
        product_events=("has_product_id", "sum"),
        score=("weight", "sum"),
        first_seen=("event_ts", "min"),
        last_seen=("event_ts", "max"),
        distinct_products=("product_id", "nunique"),
        distinct_sessions=("session_id", "nunique"),
    )
    .sort_values("score", ascending=False)
)

user_behavior_profile.head(20)


,total_events,product_events,score,first_seen,last_seen,distinct_products,distinct_sessions
account_id,,,,,,,
60a5f829c0f4e6e102e118b4,67,67,134.0,2026-08-01 04:02:57+00:00,2026-08-01 20:04:48+00:00,16,0
5ff7dde3e3b14478c7a12867,78,46,107.8,2026-08-01 08:15:06.770000+00:00,2026-08-01 22:27:56+00:00,32,1
659b8ecc6175b42337e6b384,77,42,101.8,2026-07-31 16:43:16.140000+00:00,2026-08-01 09:44:18+00:00,26,1
692998ef26c36fdc1861e0a1,44,44,88.0,2026-08-01 04:03:29+00:00,2026-08-01 20:05:32+00:00,14,0
601269e7c2128c9424f54f9d,40,32,81.8,2026-08-01 01:24:56.224000+00:00,2026-08-01 20:05:07+00:00,8,1
65431ba930304a0fa0786199,60,38,78.5,2026-08-01 02:08:46.105000+00:00,2026-08-01 11:08:22+00:00,17,1
635b9a1340e9fd0c7965be48,57,33,77.4,2026-08-01 05:40:28.234000+00:00,2026-08-01 17:02:45+00:00,10,2
648aea2be50c45cc1d397834,38,38,76.0,2026-08-01 04:02:55+00:00,2026-08-01 20:03:10+00:00,10,0
643beb32910bcf0ea9795753,63,34,75.6,2026-08-01 14:39:24.575000+00:00,2026-08-01 23:19:45+00:00,17,1


In [27]:
user_behavior_profile.reset_index(inplace = True)

In [28]:
user_behavior_profile[user_behavior_profile['account_id'] == '67a2437aee01f1d0a24c225c']

,account_id,total_events,product_events,score,first_seen,last_seen,distinct_products,distinct_sessions


In [29]:
user_behavior_profile.shape

(31004, 8)

## Realtime Pipeline Strategy

1. Capture every marketplace behavior as an immutable event: product click, taxon click, add to cart, cart update, wishlist, order, limit/budget events, search, and recommendation impression/click.

2. Standardize the event contract before scoring. Required fields: `event_id`, `account_id`, `session_id`, `event_type`, `product_id`, `taxon_id`, `event_ts`, `source`, `context`, `p_date`. The notebook's `interactions` dataframe is the offline prototype of that contract.

3. Stream ingestion should write raw events first, then normalized events. Use raw storage for replay/debugging and normalized storage for recommendation features.

4. Maintain online features per user/session: recent product ids, recent categories/taxons, cart contents, wishlist products, order history, price affinity, brand affinity, negative signals, and last activity timestamp.

5. Generate candidates from multiple lanes: same-category products, similar products from catalog attributes, cart complements, wishlist substitutes, popular products within clicked taxons, trending products, and cold-start/editorial fallback.

6. Rank candidates with a business-aware score: user intent weight, recency decay, catalog similarity, stock/availability, price fit, diversity, margin/promotion boost if allowed, and suppression rules for purchased/unavailable/repeated impressions.

7. Post recommendations through a dedicated recommendation API or message topic. The scorer should emit `account_id`, `recommendation_id`, ranked products, reason codes, model_version, generated_at, and expiry.

8. Log impressions, clicks, add-to-cart, wishlist, and orders from recommendation surfaces. Without these feedback events, realtime personalization cannot improve safely.


## First Build Milestones

- Milestone 1: finish offline validation in this notebook by confirming product/catalog join keys and measuring coverage.
- Milestone 2: create a daily batch recommender using the same `interactions` contract and write results to a recommendation table.
- Milestone 3: add streaming normalization for new events and update online user/session features.
- Milestone 4: expose recommendations through API/topic and track recommendation impressions/clicks.
- Milestone 5: introduce ranking experiments and A/B metrics: CTR, add-to-cart rate, order conversion, revenue per session, freshness, and latency.
